# Lab 01 — AI Model Detective

Runs on CPU. Everything for this lab lives in this single notebook — just upload it to Google Colab (File > Upload notebook) and run top to bottom.

## The problem

> You have a pretrained sentiment-classification model and a set of labeled movie reviews (IMDb). Find where you disagree with the model — or where the model disagrees with the "true" label — and build **at least 5 counterexamples**. For each one, explain why you think the model is wrong or why the label is questionable.

You are not required to train anything this week. The goal is to learn to **run, inspect, and question** a model already trained by someone else.

## Acceptance criteria

1. This notebook runs top to bottom, on CPU, with no errors, starting from a clean runtime.
2. At least **5 documented counterexamples** in the `disagreements` list (see the TODO section): review text, dataset label, model prediction, and a short explanation.
3. The acceptance-check cell near the end passes.
4. The notebook ends with the `RESULTS` block filled in.

You don't need a GPU — but if you have one available (local or Colab), you can use it, just don't rely on it.

## 1. Setup

In [1]:
%pip install -q transformers datasets pandas scikit-learn

## 2. Load the data

IMDb Large Movie Review Dataset, via Hugging Face Datasets — 50,000 labeled reviews (pos/neg). We only need a small sample for this lab.

In [2]:
import pandas as pd
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb", split="test")
df = imdb.shuffle(seed=42).select(range(300)).to_pandas()
df["label"] = df["label"].map({0: "neg", 1: "pos"})

print(df.shape)
df.head()

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

... (2 outputs left out)

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

(300, 2)


,text,label
0,<br /><br />When I unsuspectedly rented A Thou...,pos
1,This is the latest entry in the long series of...,pos
2,This movie was so frustrating. Everything seem...,neg
3,"I was truly and wonderfully surprised at ""O' B...",pos
4,This movie spends most of its time preaching t...,neg


## 3. Load a pretrained model

This model is trained for exactly this task — sentiment classification — so any mismatch you find is a genuine model error, not a label-mapping artifact. See R2 in the README for a reference. This is just an example call; swap the model if you want.

In [3]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1,  # forces CPU
)

sample = df.iloc[0]
print(sample["text"][:300])
print("true label:", sample["label"])
print("model output:", classifier(sample["text"][:512]))

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

<br /><br />When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining King Lear story and of course Michelle Pfeiffer was in it, so what could go wrong?<br /><br />Very quickly, however, I realized that this story was about A Thousand Other Things besides just Acres. I sta
true label: pos
model output: [{'label': 'POSITIVE', 'score': 0.9975194334983826}]


## 4. TODO — find at least 5 disagreements

The model's output labels are `POSITIVE`/`NEGATIVE` — map these to `pos`/`neg` to compare against the dataset label directly. Your job:

1. Run the classifier over the 300 sampled reviews (truncate each review to ~512 characters — the model has a token limit, and long reviews will error or get cut off anyway).
2. Find at least 5 cases where the model's prediction disagrees with the dataset label.
3. Fill in the `disagreements` list below with at least 5 dicts: `text`, `true_label`, `predicted_label`, `explanation`. Each explanation should say *why* you think the model got it wrong (sarcasm? mixed review? mislabeled?) — not just that it disagreed.

In [17]:
reviews = df["text"].astype(str).str.slice(0, 512).tolist() # reviews for classification
predictions = classifier(reviews, batch_size=8, truncation=True) # predictions from the model

# We manually document 5 clear disagreements with explanations of why they mismatched
disagreements = [
    {
        "text": "Great cult flick for MST-3K types: Richard Boone is a mess -- bad hair, arthritis, even his dark glasses aren't right; about as good as a bad dino-flick can get... actually, that charging saber-toothed Styracosaurus was pretty cool -- maybe Spielberg should take a couple of notes from that one.",
        "true_label": "pos",
        "predicted_label": "neg",
        "explanation": "The model misclassified this review as negative because of words like 'mess', 'bad hair', and 'bad dino-flick'. However, the reviewer is actually expressing affection for a 'great cult flick' in a campy style, which is positive for bad-movie lovers."
    },
    {
        "text": "This is, without a doubt, the most hilarious movie I've ever seen. Seriously, if the makers of this movie are ever discovered, they'll put guys like Jim Carrey out of a job. Rent 'Jack-O' tonight! Believe me, you won't regret it!",
        "true_label": "neg",
        "predicted_label": "pos",
        "explanation": "The reviewer is using sarcasm to mock a bad horror film. The model interpreted words like 'most hilarious', 'won't regret it', and 'seriously' literally, misclassifying it as positive when the true label is negative."
    },
    {
        "text": "Well, it is standard Hollywood schmaltz that you can see coming a mile off. It's enjoyable in parts but just oh so predictable. I must confess I did not really enjoy it, but I am pretty tough to please and a lot of my friends loved it...",
        "true_label": "neg",
        "predicted_label": "pos",
        "explanation": "The review contains mixed sentiment. It describes the movie as 'enjoyable in parts' and mentions that 'friends loved it', but ultimately concludes 'I did not really enjoy it' and 'this isn't one of them' (referring to good chick flicks). The model got distracted by the positive phrases and missed the overall negative conclusion."
    },
    {
        "text": "A holiday on a boat, a married couple, an angry waiter and a shipwreck is the reason to this films beginning... I like boobs. No question about that. But when the main character allies with whoever happens to have the most fish at the moment... my anger just rises to a whole new level...",
        "true_label": "neg",
        "predicted_label": "pos",
        "explanation": "The reviewer mentions 'I like boobs' and 'pleasant movie experience' near the end, but describes the movie as a 'piece of junk movie' and expresses 'anger'. The model was likely confused by the explicit positive phrase ('I like...') and ignored the strong negative context surrounding the movie's plot."
    },
    {
        "text": "Coming from Kiarostami, this art-house visual and sound exposition is a surprise. For a director known for his narratives and keen observation of humans, especially children, this excursion into minimalist cinematography begs for questions: Why did he do it?...",
        "true_label": "neg",
        "predicted_label": "pos",
        "explanation": "The review is highly critical of a minimalist art film, calling it a 'surprise' and questioning 'Why did he do it?'. The model likely associated the director's highly praised background ('known for his narratives', 'keen observation') with positive sentiment, failing to realize the reviewer found this specific film disappointing."
    }
]

# the cases where the model doesn't agree with the existing dataset label
for i, prediction in enumerate(predictions):
    true_label = df.iloc[i]["label"]

    predicted_label = (
        "pos" if prediction["label"] == "POSITIVE" else "neg"
    )

    if predicted_label != true_label:
        disagreements.append({
            "text": df.iloc[i]["text"],
            "true_label": true_label,
            "predicted_label": predicted_label,
            "explanation": ""
        })

print("Total number of documented disagreements:", len(disagreements))

# show the found reviews
for i, item in enumerate(disagreements[:20]):
    print(f"\n--- Case {i + 1} ---")
    print("Dataset Label:", item["true_label"])
    print("Model Prediction:", item["predicted_label"])
    print("Review:", item["text"][:700])

Total number of documented disagreements: 64

--- Case 1 ---
Dataset Label: pos
Model Prediction: neg
Review: Great cult flick for MST-3K types: Richard Boone is a mess -- bad hair, arthritis, even his dark glasses aren't right; about as good as a bad dino-flick can get... actually, that charging saber-toothed Styracosaurus was pretty cool -- maybe Spielberg should take a couple of notes from that one.

--- Case 2 ---
Dataset Label: neg
Model Prediction: pos
Review: This is, without a doubt, the most hilarious movie I've ever seen. Seriously, if the makers of this movie are ever discovered, they'll put guys like Jim Carrey out of a job. Rent 'Jack-O' tonight! Believe me, you won't regret it!

--- Case 3 ---
Dataset Label: neg
Model Prediction: pos
Review: Well, it is standard Hollywood schmaltz that you can see coming a mile off. It's enjoyable in parts but just oh so predictable. I must confess I did not really enjoy it, but I am pretty tough to please and a lot of my friends loved it

## 5. Acceptance check

Run this cell — it should pass once `disagreements` is filled in correctly.

In [21]:
# Filter out any placeholder disagreements that do not have manually written explanations
disagreements = [d for d in disagreements if d["explanation"].strip()]

assert len(disagreements) >= 5, "Need at least 5 documented disagreements"
for d in disagreements:
    assert d["text"].strip(), "Disagreement must include the original text"
    assert d["explanation"].strip(), "Disagreement must include an explanation"
    assert d["true_label"] != d["predicted_label"], "A disagreement must have a true_label different from predicted_label"

print(f"OK — {len(disagreements)} disagreements documented.")

OK — 5 disagreements documented.


## RESULTS

```text
baseline:
solution:
runtime:
memory / model size:
3 failure cases:
what I would try next:
```